# envs

In [1]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn
import numpy as np
from torch import optim
from torch.autograd import Variable
from torch.functional import F
# from net import Net1, VNet
from net import *
import argparse
from sklearn.metrics import confusion_matrix
import warnings
from sklearn.model_selection import train_test_split

from imblearn import over_sampling, under_sampling,combine
from compare_model import *

from net import *
from utils import *

np.random.seed(0)
seed_torch()
warnings.filterwarnings("ignore")
torch.manual_seed(123)

# functions

In [3]:
def build_model(features):
    model = Net2(features)
    return model

def meta_model_update2(meta_model_1, 
                       vnet_1, 
                       x_1, y_1,
                       epoch, lmbda, lmbda2):
    outputs = meta_model_1(x_1).squeeze(-1)
    outputs = torch.where(torch.isnan(outputs), torch.zeros_like(outputs), outputs)
    cost = nn.BCELoss(reduction='none')
    cost = cost(outputs, y_1)
    cost_v = torch.reshape(cost, (len(cost), 1))
    v_lambda = vnet_1(cost_v.data)
    w1 = abs((v_lambda - v_lambda.max()) / (v_lambda.max() - v_lambda.min()))

    l_f_meta_1 = torch.sum(cost_v * w1) + lmbda * F.l1_loss(meta_model_1.linear.weight,
                                                           target=torch.zeros_like(meta_model_1.linear.weight.detach()),
                                                           size_average=False) + lmbda2 * torch.sum(torch.pow(meta_model_1.linear.weight, 2))

    meta_model_1.zero_grad()
    grads = torch.autograd.grad(l_f_meta_1, (meta_model_1.params()), create_graph=True)
    meta_lr = 1e-3
    meta_model_1.update_params(lr_inner=meta_lr, source_params=grads)
    del grads

def train_auto_meta3(model_1, 
                     vnet_1, 
                     optimizer_model_1, 
                     optimizer_vnet_1, 
                     x_1, y_1, 
                     mx_1, my_1, epoch, lmbda, lmbda2):
    featureDim = x_1.shape[1]
    model_1.train()
    my_1 = torch.tensor(my_1)
    y_1 = torch.tensor(y_1)

    ###########################################step 1#################################
    # meta_model_1 = MultiLinearRegression1(featureDim, 1)
    meta_model_1 = build_model(featureDim)
    meta_model_1.load_state_dict(model_1.state_dict())

    meta_model_update2(meta_model_1, vnet_1, x_1, y_1, epoch, lmbda, lmbda2)

    ###########################################step 2#################################
    mx_1 = mx_1.float()
    y_g_hat = meta_model_1(mx_1).squeeze(-1)
    y_g_hat = torch.where(torch.isnan(y_g_hat), torch.zeros_like(y_g_hat), y_g_hat)
    cost = nn.BCELoss(reduction='none')
    l_g_meta1 = cost(y_g_hat, my_1.to(torch.float32))
    
    l_g_meta1 = l_g_meta1 + lmbda* F.l1_loss(meta_model_1.linear.weight,
                                          target=torch.zeros_like(meta_model_1.linear.weight.detach()),
                                          size_average=True)  + lmbda2 * torch.sum(torch.pow(meta_model_1.linear.weight, 2))

    optimizer_vnet_1.zero_grad()
    l_g_meta1.sum().backward()
    optimizer_vnet_1.step()

    ###########################################step 3#################################
    loss_1, w1 = calculate_weight_loss2(model_1, vnet_1, x_1, y_1)

    loss_1 = loss_1 + lmbda * F.l1_loss(model_1.linear.weight,
                                       target=torch.zeros_like(model_1.linear.weight.detach()),
                                       size_average=False) + lmbda2 * torch.sum(torch.pow(model_1.linear.weight, 2))

    optimizer_model_1.zero_grad()
    loss_1.sum().backward()
    optimizer_model_1.step()

def load_data_val_stim(X_train, y_train, folds_num=5):
    skf = StratifiedKFold(n_splits=folds_num)
    folds = []

    for fold_index, (train_index, val_index) in enumerate(skf.split(X_train, y_train)):
        X_fold_train, X_fold_val = X_train[train_index], X_train[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        
        folds.append({
            'train': (X_fold_train, y_fold_train),
            'val': (X_fold_val, y_fold_val)
        })
        print(f"Fold {fold_index + 1}:")
        print("X_fold_train set shape:", X_fold_train.shape)
        print("y_fold_train set distribution:", Counter(y_fold_train))
        print("X_fold_val set shape:", X_fold_val.shape)
        print("y_fold_val set distribution:", Counter(y_fold_val))

    return folds

def model_train(X_train, y_train, num_epochs, lr, lmbda, lmbda2):
    sampleNum = X_train.shape[0]
    featureDim = X_train.shape[1]
    meta_data_size = int(sampleNum*0.1) + 1  # the metadata size, set to: 0.1 / 0.2 / 0.3
    meta_x = np.concatenate((X_train[0:meta_data_size], X_train[sampleNum-meta_data_size:sampleNum]), axis=0)
    meta_x = torch.tensor(meta_x)
    meta_y = np.concatenate((y_train[0:meta_data_size], y_train[sampleNum-meta_data_size:sampleNum]), axis=0)

    vnet_1 = VNet(1, 300, 1)
    model_1 = build_model(featureDim)

    criterion_1 = nn.BCELoss(reduction='none')
    optimizer_1 = optim.SGD(model_1.params(), lr=1e-3)

    criterion_vnet_1 = nn.BCELoss(reduction='none')
    optimizer_vnet_1 = torch.optim.Adam(vnet_1.parameters(), 1e-2)

    loss = nn.BCELoss(reduction='none')
    W = torch.normal(0, 0.01, size=(featureDim, 1), requires_grad=True)

    batch_size = 10
    batch_size_meta = 10 
    start = 0
    for epoch in range(num_epochs):
        # 迭代训练数据
        for batch_indices in data_iter(batch_size, X_train, y_train):
            x_subset = X_train[batch_indices]  
            y_subset = y_train[batch_indices]  
            y_subset = torch.FloatTensor(y_subset).squeeze(-1) 
            meta_x_subset, meta_y_subset = next(data_iter_meta(batch_size_meta, meta_x, meta_y))  

            train_auto_meta3(model_1, 
                            vnet_1, 
                            optimizer_1, 
                            optimizer_vnet_1, 
                            x_subset, y_subset,
                            meta_x_subset, meta_y_subset, 
                            epoch, lmbda, lmbda2)

            yhat = sigmoid_net(x_subset, W)
            yhat = yhat.squeeze(-1)
            l = loss(yhat, y_subset) + lmbda * F.l1_loss(W, target=torch.zeros_like(W.detach()),
                                                        size_average=False)

            l.sum().backward()
            updater(x_subset.shape[0], W, 1e-3)

            if start == 0:
                start = 1
            else:
                with torch.no_grad():
                    model_1.linear.weight[torch.where(torch.abs(model_1.linear.weight) <= lmbda * lr)] = 0

    Vnet_test1 = vnet_1(torch.tensor([0.1]).unsqueeze(1).float())
    Vnet_test2 = vnet_1(torch.tensor([100]).unsqueeze(1).float())

    if  Vnet_test1 < Vnet_test2:
        vnet_trend = 'up'
    else:
        vnet_trend = 'down'

    return model_1, vnet_1, vnet_trend


# Args set

In [3]:
class GetArgs():
    def __init__(self):
        self.num_features = 600
        self.num_examples = 100
        self.zero_ratio = 0
        self.sigma = 2
        self.IR = 1.5
        self.set_seed = 42
        
        self.repeat = 1
        self.lr1 = 0.01
        self.lr2 = 0.1
        self.lmbda = 0.01
        self.lmbda2 = 0.01
        self.num_epochs = 20
args = GetArgs()

lr = 0.05
nums = 1
lmbda = args.lmbda
lmbda2 = args.lmbda2
num_epochs = args.num_epochs
featureDim = args.num_features
sampleNum = args.num_examples
zero_ratio = args.zero_ratio
sigma = args.sigma
ir = args.IR
set_seed = args.set_seed

# Data split

In [4]:
true_w_1 = torch.zeros(featureDim)
true_w_1[0:9] = 1
true_w_1[10:15] = -1
true_w_1[16:19] = 2

X_train, y_train = synthetic_binary_data(true_w_1, featureDim, sampleNum, 0, sigma, ir, set_seed)
X_test, y_test = synthetic_binary_data(true_w_1, featureDim, 50, 0, sigma, 1, set_seed)
print(X_train.shape)
print(X_test.shape)

folds_num = 5
train_folds = load_data_val_stim(X_train, y_train, folds_num)

torch.Size([100, 600])
torch.Size([50, 600])
Fold 1:
X_fold_train set shape: torch.Size([80, 600])
y_fold_train set distribution: Counter({0: 48, 1: 32})
X_fold_val set shape: torch.Size([20, 600])
y_fold_val set distribution: Counter({0: 12, 1: 8})
Fold 2:
X_fold_train set shape: torch.Size([80, 600])
y_fold_train set distribution: Counter({0: 48, 1: 32})
X_fold_val set shape: torch.Size([20, 600])
y_fold_val set distribution: Counter({0: 12, 1: 8})
Fold 3:
X_fold_train set shape: torch.Size([80, 600])
y_fold_train set distribution: Counter({0: 48, 1: 32})
X_fold_val set shape: torch.Size([20, 600])
y_fold_val set distribution: Counter({0: 12, 1: 8})
Fold 4:
X_fold_train set shape: torch.Size([80, 600])
y_fold_train set distribution: Counter({0: 48, 1: 32})
X_fold_val set shape: torch.Size([20, 600])
y_fold_val set distribution: Counter({0: 12, 1: 8})
Fold 5:
X_fold_train set shape: torch.Size([80, 600])
y_fold_train set distribution: Counter({0: 48, 1: 32})
X_fold_val set shape: torc

In [5]:
print("train y:", Counter(y_train))

train y: Counter({0: 60, 1: 40})


# train

In [ ]:
best_prec1 = 0
better_trend_up = 0
better_trend_dw = 0
for i in range(folds_num):
    X_train_fold, y_train_fold = train_folds[i]['train']
    X_train_fold = torch.tensor(X_train_fold).float()

    X_val_fold, y_val_fold = train_folds[i]['val']
    X_val_fold = torch.tensor(X_val_fold).float()

    model_1, vnet_1, vnet_trend_1 = model_train(X_train_fold, y_train_fold, num_epochs, lr, lmbda, lmbda2)
    vnet_trend_2 = vnet_trend_1
    train_times = 0
    while vnet_trend_1 == vnet_trend_2 and train_times < 50:
        model_2, vnet_2, vnet_trend_2 = model_train(X_train_fold, y_train_fold, num_epochs, lr, lmbda, lmbda2)
        train_times+=1

    if train_times < 50:
        best_model = model_select(model_1, model_2, X_val_fold, y_val_fold)
        if best_model == 0:
            vnet_trend = vnet_trend_1
        else:
            vnet_trend = vnet_trend_2
    else:
        vnet_trend = vnet_trend_1
    
    if vnet_trend == 'up':
        better_trend_up += 1
    if vnet_trend == 'down':
        better_trend_dw += 1

if better_trend_up >= better_trend_dw:
    better_trend = 'up'
else:   
    better_trend = 'down'
vnet_trend = 'nan'
while vnet_trend != better_trend:
    model, vnet, vnet_trend = model_train(X_train, y_train, num_epochs, lr, lmbda, lmbda2)

with torch.no_grad():
    y_test_pred = norY(model(X_test).squeeze(-1).detach().numpy()) 
    auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy = print_eva(y_test, y_test_pred, model(X_test).squeeze(-1).detach().numpy(), 'test')
    print('auc: {:.3f}\t'. format(auc))
if auc > best_prec1:
    best_lambda = lmbda
    best_lambda2 = lmbda2
    aucb, accb, senb, speb, gmeanb, f1_scoreb, AUPRCb, MCCb, balanced_accuracyb  = auc, acc, sen, spe, gmean, f1_score, AUPRC, MCC, balanced_accuracy
    best_model_all = model
    best_vnet_all = vnet
    best_vnet_trend = vnet_trend

print('AUC: {:.3f}\t ACC: {:.3f}\t SEN: {:.3f}\t SPE:{:.3f}\t Gmean:{:.3f}\t f1score:{:.3f}\t AUPRC:{:.3f}\t MCC:{:.3f}\t balanced_accuracy:{:.3f}\t'.
    format(aucb, accb, senb, speb, gmeanb, f1_scoreb, AUPRCb, MCCb, balanced_accuracyb))

# Coefs

In [ ]:
beta = best_model_all.linear.weight.detach().squeeze()

with torch.no_grad():
    y_test_pred = norY(best_model_all(X_test).squeeze(-1).detach().numpy()) 
    auc, acc, sen, spe,  gmean, f1_score = print_eva(y_test, y_test_pred, best_model_all(X_test).squeeze(-1).detach().numpy(), 'test')
    # print('[{0}]\t {1:.3f}\t {2:.3f}\t {3:.3f}\t {4:.3f}\t {5:.3f}\t {6:.3f}\t'.
    #     format(epoch, auc, acc, sen, spe,  gmean, f1_score))

    outputs = best_model_all(X_train).squeeze(-1)
    outputs = torch.where(torch.isnan(outputs), torch.zeros_like(outputs), outputs)
    cost = nn.BCELoss(reduction='none')
    y1_2 = torch.FloatTensor(y_train).squeeze(-1)
    cost = cost(outputs, torch.tensor(y1_2)) 
    cost_v = torch.reshape(cost, (len(cost), 1)).detach()
    v_weght = vnet_1(cost_v)
    v_weght_new = abs((v_weght - v_weght.max()) / (v_weght.max() - v_weght.min()))
    merged_tensor = torch.cat((cost_v, v_weght_new), dim=1)
    numpy_array = merged_tensor.numpy()
    w_v_df = pd.DataFrame(numpy_array, columns=['loss', 'weight_v'])

sorted_indices = torch.argsort(torch.abs(beta), descending=True)
non_zero_indices = torch.nonzero(beta)[:, 0]
zero_indices = torch.nonzero(torch.eq(beta, 0))[:, 0]
sorted_weights = torch.gather(beta, 0, sorted_indices)
non_zero_weights = torch.gather(beta, 0, non_zero_indices)
zero_weights = torch.gather(beta, 0, zero_indices)

abs_tensor = torch.abs(beta)
top_values, top_indices = torch.topk(abs_tensor, k=20)
print(np.sum(top_indices.numpy() < 20))
print(len(non_zero_weights))

# Compare method

In [74]:
x_resampled, y_resampled = over_sampling.SMOTE().fit_resample(X_train, y_train)

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'elasticnet')

In [ ]:
Compare_main_pip_cv(x_resampled, y_resampled, X_test, y_test, model_meth = 'elasticnet')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'elasticnetCS')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'RF')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'SVM')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'KNN')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'XGBoost')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'AdaBoost')

In [ ]:
Compare_main_pip_cv(X_train, y_train, X_test, y_test, model_meth = 'LightGBM')